---
layout: post
courses: { csa: {week: 19} }
toc: true
codemirror: true
title: 2022 FRQ 4
year: 2022
frq_number: 4
unit: [Big Idea Unit Here i.e. Unit 1]
category:  2D Array
description: "AP CSA {{ page.year }} FRQ #{{ page.frq_no }} - {{ page.unit }} - {{ page.category }}"
permalink: /csa/frqs/2022/4
author: Aashray Rajagopalan
---

# 2022 FRQ 4 — Grid Quest UI Game

This lesson turns **AP CSA 2022 FRQ 4** into a **Java UI game**.

## Main concept
**2D array traversal**
- Part A: fill every cell in a grid with valid random values
- Part B: scan each column and count how many are increasing

## How this notebook works
- The **actual game is entirely in Java**
- The game uses a **Swing UI**
- There is only **one popcorn hack**, and it is built into the game as an interactive quiz
- The game teaches the FRQ by letting the player:
  - generate valid grid values
  - inspect rows vs. columns
  - predict increasing columns
  - play a boss battle based on the FRQ logic


## FRQ Prompt

You are given this partial class:

```java
public class Data
{
    public static final int MAX = /* value not shown */;
    private int[][] grid;

    public void repopulate()
    { /* to be implemented in part (a) */ }

    public int countIncreasingCols()
    { /* to be implemented in part (b) */ }
}
```

### Part A — `repopulate`
Fill every element of `grid` with a random value such that:
- the value is between `1` and `MAX`, inclusive
- the value is divisible by `10`
- the value is **not** divisible by `100`

All valid values must have an equal chance of being generated.

### Part B — `countIncreasingCols`
Return the number of columns in `grid` that are in increasing order.

A column is increasing if every value after the first row is **greater than or equal to** the value above it.


In [ ]:
// CODE_RUNNER: Grid Quest Java UI for AP CSA 2022 FRQ 4
import javax.swing.*;
import javax.swing.border.*;
import java.awt.*;
import java.util.Random;

public class GridQuestMain {

    static class Data {
        public static final int MAX = 500;
        private int[][] grid;
        private final Random rand = new Random();

        public Data(int rows, int cols) {
            grid = new int[rows][cols];
        }

        public Data(int[][] grid) {
            this.grid = grid;
        }

        public void repopulate() {
            int maxMultiple = MAX / 10;
            for (int r = 0; r < grid.length; r++) {
                for (int c = 0; c < grid[0].length; c++) {
                    int value;
                    do {
                        int k = rand.nextInt(maxMultiple) + 1;
                        value = 10 * k;
                    } while (value % 100 == 0);
                    grid[r][c] = value;
                }
            }
        }

        public int countIncreasingCols() {
            int count = 0;
            for (int c = 0; c < grid[0].length; c++) {
                boolean increasing = true;
                for (int r = 1; r < grid.length; r++) {
                    if (grid[r][c] < grid[r - 1][c]) {
                        increasing = false;
                        break;
                    }
                }
                if (increasing) {
                    count++;
                }
            }
            return count;
        }

        public int[][] getGrid() {
            return grid;
        }

        public boolean isValidValue(int value) {
            return value >= 1 && value <= MAX && value % 10 == 0 && value % 100 != 0;
        }

        public String validityReport() {
            StringBuilder sb = new StringBuilder("Rule check:\n");
            for (int r = 0; r < grid.length; r++) {
                for (int c = 0; c < grid[0].length; c++) {
                    int value = grid[r][c];
                    sb.append("grid[").append(r).append("][").append(c).append("] = ").append(value)
                      .append(isValidValue(value) ? "  ✅ valid\n" : "  ❌ invalid\n");
                }
            }
            return sb.toString();
        }

        public String columnReport() {
            StringBuilder sb = new StringBuilder();
            for (int c = 0; c < grid[0].length; c++) {
                boolean increasing = true;
                sb.append("Column ").append(c).append(": ");
                for (int r = 0; r < grid.length; r++) {
                    sb.append(grid[r][c]);
                    if (r < grid.length - 1) sb.append(" -> ");
                }
                for (int r = 1; r < grid.length; r++) {
                    if (grid[r][c] < grid[r - 1][c]) {
                        increasing = false;
                        break;
                    }
                }
                sb.append(increasing ? "   ✅ increasing\n" : "   ❌ drops somewhere\n");
            }
            sb.append("Total increasing columns = ").append(countIncreasingCols());
            return sb.toString();
        }
    }

    static class GridQuestUI extends JFrame {
        private final CardLayout cards = new CardLayout();
        private final JPanel cardPanel = new JPanel(cards);

        private final JLabel missionLabel = new JLabel("Mission 1 of 5", SwingConstants.CENTER);
        private final JLabel conceptLabel = new JLabel("Concept: arrays store data by row and column", SwingConstants.CENTER);
        private final JLabel scoreLabel = new JLabel("Points: 0", SwingConstants.CENTER);

        private int score = 0;
        private int stage = 1;

        private boolean stage1Done = false;
        private boolean popcornDone = false;
        private boolean stage3Done = false;
        private boolean stage4Done = false;
        private boolean bossDone = false;

        private final Data repopData = new Data(4, 5);
        private final Data bossData = new Data(4, 4);
        private final Data columnData = new Data(new int[][]{
                {10, 50, 40, 20},
                {20, 40, 40, 25},
                {30, 60, 30, 25},
                {40, 80, 90, 24}
        });

        private final JPanel repopGridPanel = new JPanel();
        private final JPanel bossGridPanel = new JPanel();
        private final JPanel coordinateGridPanel = new JPanel();
        private final JPanel columnGridPanel = new JPanel();

        private final JLabel stage1Feedback = new JLabel("Click the target coordinate to learn how grid[row][col] works.");
        private final JLabel popcornFeedback = new JLabel("Choose the best traversal for repopulate().");
        private final JLabel repopFeedback = new JLabel("Press Generate Grid, then inspect whether every tile follows the rules.");
        private final JLabel columnFeedback = new JLabel("Predict how many columns are increasing, then analyze the column report.");
        private final JLabel bossFeedback = new JLabel("Final mission: use both ideas together for points.");

        private final JTextArea repopReport = makeOutputArea();
        private final JTextArea columnReport = makeOutputArea();
        private final JTextArea bossReport = makeOutputArea();

        private final JTextArea solutionA = makeCodeArea(getPartASolution());
        private final JTextArea solutionB = makeCodeArea(getPartBSolution());

        GridQuestUI() {
            setTitle("Grid Quest - AP CSA 2022 FRQ 4");
            setSize(1240, 860);
            setMinimumSize(new Dimension(1180, 800));
            setDefaultCloseOperation(JFrame.EXIT_ON_CLOSE);
            setLocationRelativeTo(null);
            setLayout(new BorderLayout(12, 12));
            ((JPanel) getContentPane()).setBorder(new EmptyBorder(12, 12, 12, 12));

            add(buildHeader(), BorderLayout.NORTH);

            cardPanel.add(buildIntro(), "intro");
            cardPanel.add(buildArraysStage(), "arrays");
            cardPanel.add(buildPopcornStage(), "popcorn");
            cardPanel.add(buildRepopulateStage(), "repopulate");
            cardPanel.add(buildColumnsStage(), "columns");
            cardPanel.add(buildBossStage(), "boss");
            cardPanel.add(buildEndStage(), "end");

            add(cardPanel, BorderLayout.CENTER);
            showStage("intro", 1, "Concept: 2D arrays use coordinates like grid[row][col]");
        }

        private JPanel buildHeader() {
            JPanel header = new JPanel();
            header.setLayout(new BoxLayout(header, BoxLayout.Y_AXIS));
            header.setBorder(new CompoundBorder(new LineBorder(new Color(70, 70, 90), 1, true), new EmptyBorder(12, 12, 12, 12)));

            JLabel title = new JLabel("Grid Quest: Learn 2022 FRQ 4 Through Play", SwingConstants.CENTER);
            title.setFont(new Font("SansSerif", Font.BOLD, 28));
            title.setAlignmentX(Component.CENTER_ALIGNMENT);

            missionLabel.setFont(new Font("SansSerif", Font.BOLD, 16));
            missionLabel.setAlignmentX(Component.CENTER_ALIGNMENT);
            conceptLabel.setFont(new Font("SansSerif", Font.PLAIN, 15));
            conceptLabel.setAlignmentX(Component.CENTER_ALIGNMENT);
            scoreLabel.setFont(new Font("SansSerif", Font.BOLD, 18));
            scoreLabel.setAlignmentX(Component.CENTER_ALIGNMENT);

            header.add(title);
            header.add(Box.createVerticalStrut(6));
            header.add(missionLabel);
            header.add(Box.createVerticalStrut(2));
            header.add(conceptLabel);
            header.add(Box.createVerticalStrut(6));
            header.add(scoreLabel);
            return header;
        }

        private JPanel buildIntro() {
            JPanel panel = stageBase();
            panel.add(makeHtmlLabel(
                    "<h1>Welcome, Grid Explorer</h1>"
                  + "<p>This game teaches the <b>actual concepts</b> behind AP CSA 2022 FRQ 4.</p>"
                  + "<p>You will learn:</p>"
                  + "<ul>"
                  + "<li>how a 2D array stores values by <b>row</b> and <b>column</b></li>"
                  + "<li>why <b>repopulate()</b> uses row-by-row traversal</li>"
                  + "<li>why <b>countIncreasingCols()</b> checks values down each column</li>"
                  + "</ul>"
                  + "<p>You earn points by completing missions and using the concepts correctly.</p>"),
                    BorderLayout.CENTER);
            panel.add(navigationBar(null, e -> showStage("arrays", 1,
                    "Concept: grid[row][col] means one specific tile in the 2D array")), BorderLayout.SOUTH);
            return panel;
        }

        private JPanel buildArraysStage() {
            JPanel outer = stageBase();
            JSplitPane split = splitPane();

            JPanel left = cardColumn("Mission 1: Array Coordinates Arena",
                    "Click the target tile. This teaches that a 2D array cell is addressed as grid[row][col].");
            left.add(makeHtmlLabel("<p><b>Target:</b> click <code>grid[1][2]</code>.</p>"
                    + "<p>Remember: the first number is the <b>row</b>, the second is the <b>column</b>.</p>"));
            left.add(Box.createVerticalStrut(8));

            coordinateGridPanel.setLayout(new GridLayout(3, 3, 8, 8));
            coordinateGridPanel.setMaximumSize(new Dimension(380, 380));
            coordinateGridPanel.setBorder(new EmptyBorder(8, 0, 8, 0));
            for (int r = 0; r < 3; r++) {
                for (int c = 0; c < 3; c++) {
                    final int row = r;
                    final int col = c;
                    JButton tile = new JButton("[" + r + "][" + c + "]");
                    tile.setFont(new Font("Monospaced", Font.BOLD, 18));
                    tile.setPreferredSize(new Dimension(110, 90));
                    tile.addActionListener(e -> handleCoordinateChoice(row, col));
                    coordinateGridPanel.add(tile);
                }
            }
            JPanel gridWrap = new JPanel();
            gridWrap.add(coordinateGridPanel);
            left.add(gridWrap);
            left.add(Box.createVerticalStrut(6));
            stage1Feedback.setFont(new Font("SansSerif", Font.PLAIN, 15));
            left.add(wrapLabel(stage1Feedback));

            JPanel right = cardColumn("What this teaches",
                    "FRQ 4 is impossible to do well unless you understand how to locate a cell and how loops move through the grid.");
            right.add(makeHtmlLabel(
                    "<p><b>Key idea:</b> <code>grid[r][c]</code> points to exactly one square.</p>"
                  + "<ul>"
                  + "<li><code>r</code> is the row index</li>"
                  + "<li><code>c</code> is the column index</li>"
                  + "<li>Part A visits every square</li>"
                  + "<li>Part B keeps the column fixed and moves downward</li>"
                  + "</ul>"
                  + "<p>That is the difference between general array access and column analysis.</p>"));
            split.setLeftComponent(left);
            split.setRightComponent(right);
            outer.add(split, BorderLayout.CENTER);
            outer.add(navigationBar(e -> showStage("intro", 1, "Concept: 2D arrays use coordinates like grid[row][col]"),
                    e -> showStage("popcorn", 2, "Concept: traversal order changes based on the job")), BorderLayout.SOUTH);
            return outer;
        }

        private JPanel buildPopcornStage() {
            JPanel outer = stageBase();
            JSplitPane split = splitPane();

            JPanel left = cardColumn("Mission 2: Popcorn Hack",
                    "Only one popcorn hack in this lesson. Use it to learn the traversal idea before the FRQ methods.");
            left.add(makeHtmlLabel("<p><b>Question:</b> which traversal best matches <code>repopulate()</code>?</p>"
                    + "<p>That method fills <b>every cell</b>, so think about the cleanest way to visit the whole grid.</p>"));
            left.add(Box.createVerticalStrut(10));
            JPanel choices = new JPanel(new GridLayout(3, 1, 8, 8));
            JButton a = bigButton("Row-major: for each row, go across the columns");
            JButton b = bigButton("Column-major: for each column, go down the rows");
            JButton c = bigButton("Random jumps to different cells");
            a.addActionListener(e -> handlePopcorn("row"));
            b.addActionListener(e -> handlePopcorn("col"));
            c.addActionListener(e -> handlePopcorn("random"));
            choices.add(a); choices.add(b); choices.add(c);
            left.add(choices);
            left.add(Box.createVerticalStrut(10));
            popcornFeedback.setFont(new Font("SansSerif", Font.PLAIN, 15));
            left.add(wrapLabel(popcornFeedback));

            JPanel right = cardColumn("Why this matters for FRQ 4",
                    "The two methods in the FRQ use different movement patterns through the same 2D array.");
            right.add(makeHtmlLabel(
                    "<ul>"
                  + "<li><b>repopulate()</b>: fill every tile, so row-major traversal is natural and simple.</li>"
                  + "<li><b>countIncreasingCols()</b>: keep a column fixed and compare vertically, so the outer loop should be columns.</li>"
                  + "</ul>"
                  + "<p>That difference is the heart of this FRQ.</p>"));
            split.setLeftComponent(left);
            split.setRightComponent(right);
            outer.add(split, BorderLayout.CENTER);
            outer.add(navigationBar(e -> showStage("arrays", 1, "Concept: grid[row][col] means one specific tile in the 2D array"),
                    e -> showStage("repopulate", 3, "Concept: repopulate() fills every cell with a valid random value")), BorderLayout.SOUTH);
            return outer;
        }

        private JPanel buildRepopulateStage() {
            JPanel outer = stageBase();
            JSplitPane split = splitPane();

            JPanel left = cardColumn("Mission 3: Reactor Repopulation",
                    "Generate valid numbers exactly like Part A of the FRQ, then inspect why they count as valid.");
            left.add(makeHtmlLabel("<p><b>Rules for each generated value:</b></p>"
                    + "<ul><li>between 1 and MAX</li><li>divisible by 10</li><li>not divisible by 100</li></ul>"));
            left.add(Box.createVerticalStrut(8));
            repopGridPanel.setLayout(new GridLayout(4, 5, 6, 6));
            repopGridPanel.setBorder(new EmptyBorder(4, 0, 4, 0));
            refreshGridPanel(repopGridPanel, repopData.getGrid(), false);
            JPanel wrap = new JPanel(new BorderLayout());
            wrap.add(repopGridPanel, BorderLayout.CENTER);
            wrap.setMaximumSize(new Dimension(480, 260));
            left.add(wrap);
            left.add(Box.createVerticalStrut(8));
            JPanel buttons = new JPanel(new FlowLayout(FlowLayout.LEFT, 8, 0));
            JButton generate = bigButton("Generate Grid");
            generate.addActionListener(e -> {
                repopData.repopulate();
                refreshGridPanel(repopGridPanel, repopData.getGrid(), true);
                repopReport.setText("Grid generated. Now inspect the rules.\n\n" + repopData.validityReport());
                if (!stage3Done) {
                    addPoints(20);
                    stage3Done = true;
                }
                repopFeedback.setText("Nice. The grid is filled row by row, and each tile now has a valid Part A value.");
            });
            buttons.add(generate);
            left.add(buttons);
            left.add(Box.createVerticalStrut(8));
            left.add(wrapLabel(repopFeedback));
            left.add(Box.createVerticalStrut(8));
            left.add(new JScrollPane(repopReport));

            JPanel right = cardColumn("Visible Java solution: Part A",
                    "The actual FRQ solution is shown here while you play.");
            right.add(new JScrollPane(solutionA));
            right.add(Box.createVerticalStrut(8));
            right.add(makeHtmlLabel("<p><b>Concept connection:</b> the nested loops teach normal 2D array traversal."
                    + " The <code>do/while</code> handles the random-value rules without breaking equal chance among valid multiples of 10.</p>"));

            split.setLeftComponent(left);
            split.setRightComponent(right);
            outer.add(split, BorderLayout.CENTER);
            outer.add(navigationBar(e -> showStage("popcorn", 2, "Concept: traversal order changes based on the job"),
                    e -> showStage("columns", 4, "Concept: countIncreasingCols() scans downward in each column")), BorderLayout.SOUTH);
            return outer;
        }

        private JPanel buildColumnsStage() {
            JPanel outer = stageBase();
            JSplitPane split = splitPane();

            JPanel left = cardColumn("Mission 4: Column Climb",
                    "Predict how many columns are increasing, then inspect the vertical comparisons that Part B makes.");
            left.add(makeHtmlLabel("<p>A column is increasing if every value is <b>greater than or equal to</b> the one above it.</p>"));
            left.add(Box.createVerticalStrut(8));
            columnGridPanel.setLayout(new GridLayout(4, 4, 6, 6));
            refreshGridPanel(columnGridPanel, columnData.getGrid(), true);
            JPanel wrap = new JPanel(new BorderLayout());
            wrap.add(columnGridPanel, BorderLayout.CENTER);
            wrap.setMaximumSize(new Dimension(420, 280));
            left.add(wrap);
            left.add(Box.createVerticalStrut(8));
            JPanel controls = new JPanel(new FlowLayout(FlowLayout.LEFT, 8, 0));
            JSpinner guess = new JSpinner(new SpinnerNumberModel(0, 0, 4, 1));
            guess.setPreferredSize(new Dimension(70, 32));
            JButton analyze = bigButton("Check My Guess");
            analyze.addActionListener(e -> {
                int g = (Integer) guess.getValue();
                int actual = columnData.countIncreasingCols();
                columnReport.setText(columnData.columnReport());
                if (g == actual) {
                    columnFeedback.setText("Correct. You tracked the comparisons down each column exactly like Part B.");
                    if (!stage4Done) {
                        addPoints(25);
                        stage4Done = true;
                    }
                } else {
                    columnFeedback.setText("Not quite. Read the report and watch where a column drops as you move downward.");
                }
            });
            controls.add(new JLabel("Guess increasing columns:"));
            controls.add(guess);
            controls.add(analyze);
            left.add(controls);
            left.add(Box.createVerticalStrut(8));
            left.add(wrapLabel(columnFeedback));
            left.add(Box.createVerticalStrut(8));
            left.add(new JScrollPane(columnReport));

            JPanel right = cardColumn("Visible Java solution: Part B",
                    "The FRQ solution stays visible while the column report explains the algorithm.");
            right.add(new JScrollPane(solutionB));
            right.add(Box.createVerticalStrut(8));
            right.add(makeHtmlLabel("<p><b>Concept connection:</b> Part B is still 2D array traversal, but now the <b>outer loop is columns</b>."
                    + " The inner loop starts at row 1 because each tile is compared to the tile above it: <code>grid[r][c]</code> vs <code>grid[r-1][c]</code>.</p>"));

            split.setLeftComponent(left);
            split.setRightComponent(right);
            outer.add(split, BorderLayout.CENTER);
            outer.add(navigationBar(e -> showStage("repopulate", 3, "Concept: repopulate() fills every cell with a valid random value"),
                    e -> showStage("boss", 5, "Concept: combine generation + column checking to beat the boss")), BorderLayout.SOUTH);
            return outer;
        }

        private JPanel buildBossStage() {
            JPanel outer = stageBase();
            JSplitPane split = splitPane();

            JPanel left = cardColumn("Mission 5: Boss Battle",
                    "Use both concepts together: generate a valid grid, then predict the number of increasing columns.");
            left.add(makeHtmlLabel("<p><b>Boss rules:</b> first generate with Part A logic, then solve with Part B logic.</p>"));
            left.add(Box.createVerticalStrut(8));
            bossGridPanel.setLayout(new GridLayout(4, 4, 6, 6));
            refreshGridPanel(bossGridPanel, bossData.getGrid(), false);
            JPanel wrap = new JPanel(new BorderLayout());
            wrap.add(bossGridPanel, BorderLayout.CENTER);
            wrap.setMaximumSize(new Dimension(420, 280));
            left.add(wrap);
            left.add(Box.createVerticalStrut(8));

            JPanel controls = new JPanel(new FlowLayout(FlowLayout.LEFT, 8, 0));
            JButton spawn = bigButton("Spawn Boss Grid");
            JSpinner guess = new JSpinner(new SpinnerNumberModel(0, 0, 4, 1));
            guess.setPreferredSize(new Dimension(70, 32));
            JButton reveal = bigButton("Reveal Answer");
            spawn.addActionListener(e -> {
                bossData.repopulate();
                refreshGridPanel(bossGridPanel, bossData.getGrid(), true);
                bossReport.setText("Boss grid generated. Make your prediction, then reveal the answer.\n");
            });
            reveal.addActionListener(e -> {
                int g = (Integer) guess.getValue();
                int actual = bossData.countIncreasingCols();
                bossReport.setText(bossData.validityReport() + "\n\n" + bossData.columnReport());
                if (g == actual) {
                    bossFeedback.setText("Boss defeated. You applied both FRQ ideas correctly.");
                    if (!bossDone) {
                        addPoints(35);
                        bossDone = true;
                    }
                } else {
                    bossFeedback.setText("Close. Use the report to see where each column rises or drops.");
                }
            });
            controls.add(spawn);
            controls.add(new JLabel("Guess increasing columns:"));
            controls.add(guess);
            controls.add(reveal);
            left.add(controls);
            left.add(Box.createVerticalStrut(8));
            left.add(wrapLabel(bossFeedback));
            left.add(Box.createVerticalStrut(8));
            left.add(new JScrollPane(bossReport));

            JPanel right = cardColumn("Strategy recap",
                    "This final stage reinforces the complete FRQ flow.");
            right.add(makeHtmlLabel(
                    "<ol>"
                  + "<li>Use <b>row-major traversal</b> to fill every tile.</li>"
                  + "<li>Use the Part A value rules: valid multiples of 10, but not 100.</li>"
                  + "<li>Switch perspectives for Part B: keep the <b>column fixed</b>.</li>"
                  + "<li>Compare downward to see whether the column stays increasing.</li>"
                  + "</ol>"
                  + "<p>That is the exact conceptual story of FRQ 4.</p>"));
            split.setLeftComponent(left);
            split.setRightComponent(right);
            outer.add(split, BorderLayout.CENTER);
            outer.add(navigationBar(e -> showStage("columns", 4, "Concept: countIncreasingCols() scans downward in each column"),
                    e -> showStage("end", 6, "Concept: summarize the FRQ and your score")), BorderLayout.SOUTH);
            return outer;
        }

        private JPanel buildEndStage() {
            JPanel panel = stageBase();
            JLabel summary = makeHtmlLabel(
                    "<h2>Mission Complete</h2>"
                  + "<p><b>What the game taught:</b></p>"
                  + "<ul>"
                  + "<li>2D arrays are addressed with <code>grid[row][col]</code>.</li>"
                  + "<li>Part A teaches full-grid traversal and controlled random generation.</li>"
                  + "<li>Part B teaches column-based traversal and vertical comparison.</li>"
                  + "</ul>"
                  + "<p><b>Final score:</b> " + score + " points</p>"
                  + "<p>Use the visible solutions and the mission reports to study the FRQ again.</p>");
            panel.add(summary, BorderLayout.CENTER);
            panel.add(navigationBar(e -> showStage("boss", 5, "Concept: combine generation + column checking to beat the boss"), null), BorderLayout.SOUTH);
            return panel;
        }

        private void handleCoordinateChoice(int row, int col) {
            if (row == 1 && col == 2) {
                stage1Feedback.setText("Correct. grid[1][2] means row 1, column 2. That exact addressing skill is used all through the FRQ.");
                if (!stage1Done) {
                    addPoints(15);
                    stage1Done = true;
                }
            } else {
                stage1Feedback.setText("Not yet. Read the coordinate carefully: row first, column second. Try again for grid[1][2].");
            }
        }

        private void handlePopcorn(String choice) {
            if ("row".equals(choice)) {
                popcornFeedback.setText("Correct. repopulate() should naturally visit the whole grid row by row. Part B is the one that thinks column by column.");
                if (!popcornDone) {
                    addPoints(10);
                    popcornDone = true;
                }
            } else if ("col".equals(choice)) {
                popcornFeedback.setText("Close, but column-major fits Part B better. repopulate() just needs a clean full-grid traversal, so row-major is the usual choice.");
            } else {
                popcornFeedback.setText("Random jumps would make the logic messy. FRQ methods should use organized traversal patterns.");
            }
        }

        private void addPoints(int pts) {
            score += pts;
            scoreLabel.setText("Points: " + score);
        }

        private void showStage(String name, int missionNumber, String concept) {
            stage = missionNumber;
            missionLabel.setText("Mission " + missionNumber + " of 6");
            conceptLabel.setText(concept);
            cards.show(cardPanel, name);
        }

        private JPanel stageBase() {
            JPanel panel = new JPanel(new BorderLayout(12, 12));
            panel.setBorder(new EmptyBorder(4, 4, 4, 4));
            return panel;
        }

        private JSplitPane splitPane() {
            JSplitPane split = new JSplitPane(JSplitPane.HORIZONTAL_SPLIT);
            split.setResizeWeight(0.58);
            split.setDividerLocation(660);
            split.setBorder(null);
            return split;
        }

        private JPanel cardColumn(String title, String subtitle) {
            JPanel panel = new JPanel();
            panel.setLayout(new BoxLayout(panel, BoxLayout.Y_AXIS));
            panel.setBorder(new CompoundBorder(new LineBorder(new Color(120, 120, 140), 1, true), new EmptyBorder(14, 14, 14, 14)));
            JLabel titleLabel = new JLabel(title);
            titleLabel.setFont(new Font("SansSerif", Font.BOLD, 22));
            titleLabel.setAlignmentX(Component.LEFT_ALIGNMENT);
            JLabel subLabel = makeHtmlLabel("<p style='width:420px;'>" + subtitle + "</p>");
            subLabel.setAlignmentX(Component.LEFT_ALIGNMENT);
            panel.add(titleLabel);
            panel.add(Box.createVerticalStrut(6));
            panel.add(subLabel);
            panel.add(Box.createVerticalStrut(10));
            return panel;
        }

        private JPanel navigationBar(java.awt.event.ActionListener back, java.awt.event.ActionListener next) {
            JPanel nav = new JPanel(new FlowLayout(FlowLayout.RIGHT, 10, 4));
            if (back != null) {
                JButton b = bigButton("Back");
                b.addActionListener(back);
                nav.add(b);
            }
            if (next != null) {
                JButton n = bigButton("Next");
                n.addActionListener(next);
                nav.add(n);
            }
            return nav;
        }

        private JButton bigButton(String text) {
            JButton btn = new JButton(text);
            btn.setFont(new Font("SansSerif", Font.BOLD, 14));
            btn.setMargin(new Insets(10, 14, 10, 14));
            return btn;
        }

        private JLabel makeHtmlLabel(String html) {
            JLabel label = new JLabel("<html><body style='font-family:sans-serif; font-size:13px;'>" + html + "</body></html>");
            return label;
        }

        private JPanel wrapLabel(JLabel label) {
            JPanel p = new JPanel(new BorderLayout());
            p.add(label, BorderLayout.CENTER);
            p.setAlignmentX(Component.LEFT_ALIGNMENT);
            return p;
        }

        private void refreshGridPanel(JPanel panel, int[][] grid, boolean numbersReady) {
            panel.removeAll();
            panel.setLayout(new GridLayout(grid.length, grid[0].length, 6, 6));
            for (int r = 0; r < grid.length; r++) {
                for (int c = 0; c < grid[0].length; c++) {
                    JLabel cell = new JLabel(numbersReady ? String.valueOf(grid[r][c]) : "--", SwingConstants.CENTER);
                    cell.setOpaque(true);
                    cell.setPreferredSize(new Dimension(74, 54));
                    cell.setFont(new Font("Monospaced", Font.BOLD, 18));
                    cell.setBorder(new LineBorder(new Color(90, 90, 110), 1, true));
                    cell.setBackground(numbersReady ? new Color(245, 247, 255) : new Color(235, 235, 235));
                    panel.add(cell);
                }
            }
            panel.revalidate();
            panel.repaint();
        }

        private static JTextArea makeOutputArea() {
            JTextArea area = new JTextArea(12, 34);
            area.setFont(new Font("Monospaced", Font.PLAIN, 14));
            area.setEditable(false);
            area.setLineWrap(true);
            area.setWrapStyleWord(true);
            area.setMargin(new Insets(10, 10, 10, 10));
            area.setText("Interact with the mission to generate a report here.");
            return area;
        }

        private static JTextArea makeCodeArea(String code) {
            JTextArea area = new JTextArea(code, 20, 34);
            area.setFont(new Font("Monospaced", Font.PLAIN, 14));
            area.setEditable(false);
            area.setMargin(new Insets(10, 10, 10, 10));
            return area;
        }

        private static String getPartASolution() {
            return "public void repopulate() {\n"
                 + "    Random rand = new Random();\n"
                 + "    int maxMultiple = MAX / 10;\n\n"
                 + "    for (int r = 0; r < grid.length; r++) {\n"
                 + "        for (int c = 0; c < grid[0].length; c++) {\n"
                 + "            int value;\n"
                 + "            do {\n"
                 + "                int k = rand.nextInt(maxMultiple) + 1;\n"
                 + "                value = 10 * k;\n"
                 + "            } while (value % 100 == 0);\n\n"
                 + "            grid[r][c] = value;\n"
                 + "        }\n"
                 + "    }\n"
                 + "}";
        }

        private static String getPartBSolution() {
            return "public int countIncreasingCols() {\n"
                 + "    int count = 0;\n\n"
                 + "    for (int c = 0; c < grid[0].length; c++) {\n"
                 + "        boolean increasing = true;\n\n"
                 + "        for (int r = 1; r < grid.length; r++) {\n"
                 + "            if (grid[r][c] < grid[r - 1][c]) {\n"
                 + "                increasing = false;\n"
                 + "                break;\n"
                 + "            }\n"
                 + "        }\n\n"
                 + "        if (increasing) {\n"
                 + "            count++;\n"
                 + "        }\n"
                 + "    }\n\n"
                 + "    return count;\n"
                 + "}";
        }
    }

    public static void main(String[] args) {
        SwingUtilities.invokeLater(() -> new GridQuestUI().setVisible(true));
    }
}


GridQuestMain.main(null);


## What the game is teaching

### Stage 1 — `repopulate()`
The player sees that:
- every cell in the 2D array gets filled
- valid values are multiples of 10
- values divisible by 100 are rejected

### Popcorn Hack
The one interactive quiz checks whether the player can recognize valid values.

### Stage 3 — `countIncreasingCols()`
The player learns:
- check **columns**, not rows
- compare each value with the one above it
- count the columns that never decrease

### Boss Battle
The player combines both FRQ parts:
- generate a valid grid
- predict increasing columns
- verify the result visually


In [ ]:

## FRQ reference solutions

```java
public void repopulate()
{
    Random rand = new Random();
    int maxMultiple = MAX / 10;

    for (int r = 0; r < grid.length; r++)
    {
        for (int c = 0; c < grid[0].length; c++)
        {
            int value;
            do
            {
                int k = rand.nextInt(maxMultiple) + 1;
                value = 10 * k;
            }
            while (value % 100 == 0);

            grid[r][c] = value;
        }
    }
}

public int countIncreasingCols()
{
    int count = 0;

    for (int c = 0; c < grid[0].length; c++)
    {
        boolean increasing = true;

        for (int r = 1; r < grid.length; r++)
        {
            if (grid[r][c] < grid[r - 1][c])
            {
                increasing = false;
                break;
            }
        }

        if (increasing)
        {
            count++;
        }
    }

    return count;
}
```


Example 1 Grid:
[10, 50, 40]
[20, 40, 20]
[30, 50, 30]
countIncreasingCols() = 1

Example 2 Grid:
[10, 540, 440, 440]
[220, 450, 440, 190]
countIncreasingCols() = 2



## FRQ Question Part A

4. This question involves a two-dimensional array of integers that represents a collection of randomly generated data. A partial declaration of the Data class is shown. You will write two methods of the Data class.

public class Data
{
    public static final int MAX = /* value not shown */;
    private int[][] grid;

    /** Fills all elements of grid with randomly generated values, as described in part (a)
     *  Precondition: grid is not null.
     *                grid has at least one element.
     */
    public void repopulate()
    { /* to be implemented in part (a) */ }

    /** Returns the number of columns in grid that are in increasing order, as described
     *  in part (b)
     *  Precondition: grid is not null.
     *                grid has at least one element.
     */
    public int countIncreasingCols()
    { /* to be implemented in part (b) */ }

    // There may be instance variables, constructors, and methods that are not shown.
}


(a) Write the repopulate method, which assigns a newly generated random value to each element of grid. Each value is computed to meet all of the following criteria, and all valid values must have an equal chance of being generated.

The value is between 1 and MAX, inclusive.

The value is divisible by 10.

The value is not divisible by 100.

Complete the repopulate method.

/** Fills all elements of grid with randomly generated values, as described in part (a)
 *  Precondition: grid is not null.
 *                grid has at least one element.
 */
public void repopulate()


`// CODE_RUNNER: <challenge text>`

In [ ]:
// CODE_RUNNER: FRQ #4 (a) - repopulate()
import java.util.Arrays;
import java.util.Random;

public class Main {

    public static class Data {
        public static final int MAX = 500;
        private int[][] grid;

        public Data(int rows, int cols) {
            grid = new int[rows][cols];
        }

        /**
         * Fills all elements of grid with randomly generated values:
         * - between 1 and MAX inclusive
         * - divisible by 10
         * - NOT divisible by 100
         * All valid values have equal probability.
         */
        public void repopulate() {
            Random rand = new Random();

            int maxMultiple = MAX / 10;  // largest k such that 10*k <= MAX

            for (int r = 0; r < grid.length; r++) {
                for (int c = 0; c < grid[0].length; c++) {

                    int value;
                    do {
                        int k = rand.nextInt(maxMultiple) + 1; // 1..maxMultiple
                        value = 10 * k;
                    } while (value % 100 == 0);

                    grid[r][c] = value;
                }
            }
        }
    }


    public void driver() {
        Data d = new Data(3, 5);
        d.repopulate();

        System.out.println("Generated Grid:");
        for (int[] row : d.grid) {
            System.out.println(Arrays.toString(row));
        }
    }

    public static void main(String[] args) {
        Main m = new Main();
        m.driver();
    }
}

Main.main(null);


Generated Grid:
[360, 90, 70, 360, 110]
[350, 450, 190, 460, 120]
[310, 50, 450, 270, 270]


(b) Write the countIncreasingCols method, which returns the number of columns in grid that are in increasing order. A column is considered to be in increasing order if the element in each row after the first row is greater than or equal to the element in the previous row. A column with only one row is considered to be in increasing order.

The following examples show the countIncreasingCols return values for possible contents of grid.

The return value for the following contents of grid is 1, since the first column is in increasing order but the second and third columns are not.

10  50  40
20  40  20
30  50  30

The return value for the following contents of grid is 2, since the first and third columns are in increasing order but the second and fourth columns are not.

10   540  440  440
220  450  440  190


Complete the countIncreasingCols method.

/** Returns the number of columns in grid that are in increasing order, as described
 *  in part (b)
 *  Precondition: grid is not null.
 *                grid has at least one element.
 */
public int countIncreasingCols()

Begin your response at the top of a new page in the Free Response booklet and fill in the appropriate circle indicating the question number. If there are multiple parts to this question, write the part letter with your response.

public class Data
{
    public static final int MAX = /* value not shown */;
    private int[][] grid;

    public void repopulate()
    public int countIncreasingCols()
}



In [ ]:
// CODE_RUNNER: FRQ #4 (b) - countIncreasingCols()
import java.util.Arrays;

public class Main {

    public static class Data {
        private int[][] grid;

        public Data(int[][] g) {
            grid = g;
        }

        /**
         * Returns the number of columns in grid that are in increasing order.
         * A column is increasing if for every row r > 0:
         *   grid[r][c] >= grid[r-1][c]
         * A column with only one row is considered increasing.
         */
        public int countIncreasingCols() {
            int rows = grid.length;
            int cols = grid[0].length;
            int count = 0;

            for (int c = 0; c < cols; c++) {
                boolean increasing = true;

                for (int r = 1; r < rows; r++) {
                    if (grid[r][c] < grid[r - 1][c]) {
                        increasing = false;
                        break;
                    }
                }

                if (increasing) {
                    count++;
                }
            }

            return count;
        }
    }


    private static void printGrid(int[][] g) {
        for (int[] row : g) System.out.println(Arrays.toString(row));
    }

    public void driver() {
        int[][] example1 = {
            {10, 50, 40},
            {20, 40, 20},
            {30, 50, 30}
        };

        int[][] example2 = {
            {10, 540, 440, 440},
            {220, 450, 440, 190}
        };

        System.out.println("Example 1 Grid:");
        printGrid(example1);
        Data d1 = new Data(example1);
        System.out.println("countIncreasingCols() = " + d1.countIncreasingCols());
        System.out.println();

        System.out.println("Example 2 Grid:");
        printGrid(example2);
        Data d2 = new Data(example2);
        System.out.println("countIncreasingCols() = " + d2.countIncreasingCols());
        System.out.println();
    }

    public static void main(String[] args) {
        Main m = new Main();
        m.driver();
    }
}

Main.main(null);


Example 1 Grid:
[10, 50, 40]
[20, 40, 20]
[30, 50, 30]
countIncreasingCols() = 1

Example 2 Grid:
[10, 540, 440, 440]
[220, 450, 440, 190]
countIncreasingCols() = 2



## Scoring Guidelines Part A

**(a) repopulate**

| # | Scoring Criteria | Decision Rules | Points |
|---|-----------------|----------------|--------|
| 1 | Traverses `grid` (no bounds errors) | Responses will not earn the point if they:<br>- fail to access an element of `grid`<br>- access the elements of `grid` incorrectly<br>- use enhanced for loops without using a `grid` element inside the loop | 1 point |
| 2 | Generates a random integer in a range based on `MAX` | Responses can still earn the point even if they:<br>- assume or verify that `MAX >= 10`<br><br>Responses will not earn the point if they:<br>- fail to cast to an `int` | 1 point |
| 3 | Ensure that all produced values are divisible by 10 but not by 100 | Responses can still earn the point even if they:<br>- fail to use a loop | 1 point |
| 4 | Assigns appropriate values to all elements of `grid` (algorithm) | Responses can still earn the point even if they:<br>- assume or verify that `MAX >= 10`<br>- produce some values that are not divisible by 10 or divisible by 100, if the range and distribution are otherwise correct<br><br>Responses will not earn the point if they:<br>- use enhanced for loops and fail to maintain indices<br>- produce values that are not equally distributed<br>- produce values outside the specified range<br>- exclude values that should be considered valid (other than errors in 10/100 handling) | 1 point |


## Scoring Guidelines Part B

**(b) countIncreasingCols**

| # | Scoring Criteria | Decision Rules | Points |
|---|------------------|----------------|--------|
| 5 | Traverses `grid` in column major order (no loop header bounds errors) | Responses can still earn the point even if they:<br>- access an out-of-bounds row or column index adjacent to the edge of the grid, if the loop bounds include only valid indices<br><br>Responses will not earn the point if they:<br>- fail to access an element of `grid`<br>- access the elements of `grid` incorrectly | 1 point |
| 6 | Compares two elements in the same column of `grid` | Responses can still earn the point even if they:<br>- access elements of `grid` incorrectly<br>- access elements in nonadjacent rows<br>- compare elements with `==`<br>- compare two elements in the same row instead of the same column | 1 point |
| 7 | Determines whether a single column is in increasing order (algorithm) | Responses can still earn the point even if they:<br>- fail to reset variables in the outer loop before proceeding to the next column<br><br>Responses will not earn the point if they:<br>- fail to access all pairs of adjacent elements in a single column<br>- cause a bounds error by attempting to compare the first element of a column with a previous element or the last element of a column with a subsequent element<br>- incorrectly identify a column with at least one pair of adjacent elements in decreasing order as increasing | 1 point |
| 8 | Counts all columns that are identified as increasing (algorithm) | Responses can still earn the point even if they:<br>- detect increasing order for each row instead of each column<br>- incorrectly identify increasing columns in the inner loop<br><br>Responses will not earn the point if they:<br>- fail to initialize the counter<br>- fail to reset variables in the outer loop causing subsequent runs of the inner loop to misidentify columns | 1 point |
| 9 | Returns calculated count of increasing columns | Responses can still earn the point even if they:<br>- calculate the count incorrectly | 1 point |



## Scoring Guidelines Part A

**(a) repopulate**

| # | Scoring Criteria | Decision Rules | Points |
|---|-----------------|----------------|--------|
| 1 | Traverses `grid` (no bounds errors) | Responses will not earn the point if they:<br>- fail to access an element of `grid`<br>- access the elements of `grid` incorrectly<br>- use enhanced for loops without using a `grid` element inside the loop | 1 point |
| 2 | Generates a random integer in a range based on `MAX` | Responses can still earn the point even if they:<br>- assume or verify that `MAX >= 10`<br><br>Responses will not earn the point if they:<br>- fail to cast to an `int` | 1 point |
| 3 | Ensure that all produced values are divisible by 10 but not by 100 | Responses can still earn the point even if they:<br>- fail to use a loop | 1 point |
| 4 | Assigns appropriate values to all elements of `grid` (algorithm) | Responses can still earn the point even if they:<br>- assume or verify that `MAX >= 10`<br>- produce some values that are not divisible by 10 or divisible by 100, if the range and distribution are otherwise correct<br><br>Responses will not earn the point if they:<br>- use enhanced for loops and fail to maintain indices<br>- produce values that are not equally distributed<br>- produce values outside the specified range<br>- exclude values that should be considered valid (other than errors in 10/100 handling) | 1 point |


## Scoring Guidelines Part B

**(b) countIncreasingCols**

| # | Scoring Criteria | Decision Rules | Points |
|---|------------------|----------------|--------|
| 5 | Traverses `grid` in column major order (no loop header bounds errors) | Responses can still earn the point even if they:<br>- access an out-of-bounds row or column index adjacent to the edge of the grid, if the loop bounds include only valid indices<br><br>Responses will not earn the point if they:<br>- fail to access an element of `grid`<br>- access the elements of `grid` incorrectly | 1 point |
| 6 | Compares two elements in the same column of `grid` | Responses can still earn the point even if they:<br>- access elements of `grid` incorrectly<br>- access elements in nonadjacent rows<br>- compare elements with `==`<br>- compare two elements in the same row instead of the same column | 1 point |
| 7 | Determines whether a single column is in increasing order (algorithm) | Responses can still earn the point even if they:<br>- fail to reset variables in the outer loop before proceeding to the next column<br><br>Responses will not earn the point if they:<br>- fail to access all pairs of adjacent elements in a single column<br>- cause a bounds error by attempting to compare the first element of a column with a previous element or the last element of a column with a subsequent element<br>- incorrectly identify a column with at least one pair of adjacent elements in decreasing order as increasing | 1 point |
| 8 | Counts all columns that are identified as increasing (algorithm) | Responses can still earn the point even if they:<br>- detect increasing order for each row instead of each column<br>- incorrectly identify increasing columns in the inner loop<br><br>Responses will not earn the point if they:<br>- fail to initialize the counter<br>- fail to reset variables in the outer loop causing subsequent runs of the inner loop to misidentify columns | 1 point |
| 9 | Returns calculated count of increasing columns | Responses can still earn the point even if they:<br>- calculate the count incorrectly | 1 point |

